**1. CONVOLUTIONAL AUTOENCODERS TRAINING**

In [13]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

profiles = dh.get_k8s_resource_profiles()
print(profiles)

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

['1xrtxa5000', '1xv100', 'cpu', '1xa40-shared', '2xa40-shared', '3xa40-shared', '4xa40-shared', '5xa40-shared', '6xa40-shared', '7xa40-shared', '8xa40-shared', '1xrtx5000-shared', '2xrtx5000-shared', '1xrtxa5000-shared', '2xrtxa5000-shared', '3xrtxa5000-shared', '1xv100-shared', '2xv100-shared', '3xv100-shared', '4xv100-shared', '5xv100-shared', '6xv100-shared', '7xv100-shared', '8xv100-shared', 'cpu-shared', 'default']
Progetto: floods


**SETUP PARAMETERS**

In [ ]:
# Parametri Job   
job_name = "test_encoders_1D_v3"                                
dataset = "Test" 
test_sar = False
test_opt = True                                       
#handler = pretrain_encoders                                         

parametri = {     
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                      
    "n_images2": 4, "n_channels2": 10,  
    "output_dim": 512,                              
    "mamba": False, 
    "dataset": dataset,
    "test_sar": test_sar,                                 
    "test_opt": test_opt,
    "weights_s1": "encoder-s1-weights_train_sar_1D_v1_Standard_200",
    "weights_s2": "encoder-s2-weights_train_opt_1D_v1_Standard_200",
    "n_samples": 10,         
    "recon_channels": [2,1,0]
}                                                              

print(f"PARAMETRI: {parametri}")

# volume -> circa 400 GB dataset Standard
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "200Gi"}   
    }
]

PARAMETRI: {'patch_size': 256, 'n_images1': 4, 'n_channels1': 2, 'n_images2': 4, 'n_channels2': 10, 'output_dim': 512, 'mamba': False, 'dataset': 'Test', 'test_sar': False, 'test_opt': True, 'weights_s1': 'encoder-s1-weights_train_sar_1D_v1_Standard_200', 'weights_s2': 'encoder-s2-weights_train_opt_1D_v1_Standard_200', 'n_samples': 10, 'recon_channels': [2, 1, 0]}


**BUILD ENVIRONMENT**

In [15]:
test_train_func = project.new_function(
    name= f'encoders-Floods_{dataset}_{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="test_encoders_visual", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30", "torch==2.1.2", "matplotlib==3.10.9", "digitalhub==0.15.11", "digitalhub-runtime-python==0.15.2"]
)

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = test_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

2026-09-21 08:38:25,026 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 231495eacbfe4dfa97de642c5a0a4b9e to finish...
2026-09-21 08:38:30,033 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 231495eacbfe4dfa97de642c5a0a4b9e to finish...
2026-09-21 08:38:35,042 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 231495eacbfe4dfa97de642c5a0a4b9e to finish...
2026-09-21 08:38:40,050 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 231495eacbfe4dfa97de642c5a0a4b9e to finish...
2026-09-21 08:38:45,060 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 231495eacbfe4dfa97de642c5a0a4b9e to finish...
2026-09-21 08:38:50,068 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 231495eacbfe4dfa97de642c5a0a4b9e to finish...
2026-09-21 08:38:55,077 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 231495eacbfe4dfa97de642c5a0a4b9e to finish...
2026-09-21 08

BUILD: COMPLETED


**TRAINING**

In [16]:
# action job = avvia container, esegue script, libera risorse

run_test_encoders = test_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xv100-shared",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run test_encoders avviato: {run_test_encoders.id}")
print(run_test_encoders.status.state)
print(run_test_encoders.status.message)

2026-09-21 08:39:15,288 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 739f382e66014225abb770bfc3e5158d to finish...


2026-09-21 08:39:20,294 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 739f382e66014225abb770bfc3e5158d to finish...
2026-09-21 08:39:25,302 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 739f382e66014225abb770bfc3e5158d to finish...
2026-09-21 08:39:30,313 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 739f382e66014225abb770bfc3e5158d to finish...
2026-09-21 08:39:35,322 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 739f382e66014225abb770bfc3e5158d to finish...
2026-09-21 08:39:40,334 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 739f382e66014225abb770bfc3e5158d to finish...
2026-09-21 08:39:45,447 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 739f382e66014225abb770bfc3e5158d to finish...
2026-09-21 08:39:50,456 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 739f382e66014225abb770bfc3e5158d to finish...
2026-09-21 08

Run test_encoders avviato: 739f382e66014225abb770bfc3e5158d
COMPLETED
None
